In [10]:
# Jupyter Notebook: Notebook_1_Data_Annotation.ipynb
# ==============================================================================
# --- 1. УСТАНОВКА И ИМПОРТЫ ---
# ==============================================================================
# !pip install torch torchvision numpy trimesh[easy] pandas scikit-learn plotly pyvista opencv-python tqdm
# Важно! PyVista может потребовать доп. настройки для работы в Jupyter (Jupyter server extension).
# Если 3D-рендеры не работают, это не критично, кластеризация всё равно будет выполнена.

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import cv2 as cv
import pyvista as pv
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import normalize
import plotly.express as px
import trimesh
import re

# Импортируем архитектуры и загрузчики из вашего кода
# (Предполагается, что они находятся в файле utils.py или определены в ячейке выше)
print("Все библиотеки успешно импортированы.")

Все библиотеки успешно импортированы.


In [11]:
# --- ИСПРАВЛЕННАЯ И УЛУЧШЕННАЯ ФУНКЦИЯ ЗАГРУЗКИ STL ---
# Вставьте эту функцию в ячейку, где она была определена, заменив старую.

def load_stl_xyz_only(stl_path, num_points=1024, verbose=False):
    """
    (ИСПРАВЛЕНО) Загружает, сэмплирует, центрирует и нормализует STL с подробным выводом ошибок.
    Установите verbose=True для одного файла, чтобы увидеть детальную отладку.
    """
    try:
        # 1. Загрузка сетки
        mesh = trimesh.load(stl_path, process=True, force='mesh')
        if verbose: print(f"[{stl_path}] Шаг 1: Trimesh загрузил объект типа {type(mesh)}")

        # 2. Обработка сцены (если Trimesh загрузил сцену вместо одной сетки)
        if isinstance(mesh, trimesh.Scene):
            if verbose: print(f"[{stl_path}] -> Это сцена, объединяем геометрию...")
            mesh = mesh.dump(concatenate=True)
        
        # 3. Проверка на наличие вершин
        if not hasattr(mesh, 'vertices') or len(mesh.vertices) == 0:
            if verbose: print(f"[{stl_path}] ОШИБКА: Сетка пуста (нет вершин).")
            return None
        
        # 4. Проверка площади поверхности (важно!)
        # Если площадь очень мала или равна нулю, сэмплирование не удастся.
        if mesh.area < 1e-6:
            if verbose: print(f"[{stl_path}] ОШИБКА: Площадь поверхности сетки почти равна нулю ({mesh.area}). Невозможно сэмплировать.")
            return None

        # 5. Сэмплирование точек
        points, _ = trimesh.sample.sample_surface(mesh, num_points)
        if verbose: print(f"[{stl_path}] Шаг 2: Сэмплировано {len(points)} точек.")
        
        # 6. Проверка количества точек
        if len(points) < 1: # Даже если мы запросили много, должна быть хотя бы одна
             if verbose: print(f"[{stl_path}] ОШИБКА: Не удалось сэмплировать ни одной точки.")
             return None
        
        # 7. (БЕЗ ИЗМЕНЕНИЙ) Выравнивание количества точек до num_points
        if len(points) < num_points:
            indices = np.random.choice(len(points), num_points, replace=True)
        else:
            indices = np.random.choice(len(points), num_points, replace=False)
        points = points[indices]

        # 8. (БЕЗ ИЗМЕНЕНИЙ) Центрирование и нормализация
        centroid = np.mean(points, axis=0)
        points -= centroid
        max_dist = np.max(np.linalg.norm(points, axis=1))

        # 9. Проверка вырожденности (если все точки в одном месте)
        if max_dist < 1e-6:
            if verbose: print(f"[{stl_path}] ОШИБКА: Облако точек вырождено (все точки в одной координате).")
            return None
        
        points /= max_dist
        if verbose: print(f"[{stl_path}] -> Успешно обработано!")
        
        return points.astype(np.float32)
        
    except Exception as e:
        # 10. Отлов всех остальных ошибок
        if verbose:
            import traceback
            print(f"[{stl_path}] КРИТИЧЕСКАЯ ОШИБКА: Произошло необработанное исключение.")
            traceback.print_exc() # Печатаем полный traceback для детальной отладки
        return None

class PairedStlImageDataset(Dataset):
    """(ПЕРЕРАБОТАНО) Датасет для структуры 1 STL -> 25 изображений."""
    def __init__(self, stl_root, image_root, num_points=4096, image_size=224):
        self.num_points = num_points
        self.stl_root = Path(stl_root)
        self.image_root = Path(image_root)
        all_stl_files = sorted([f for f in self.stl_root.rglob('*.stl') if f.is_file()])
        
        self.paired_files = []
        # Паттерн для поиска 4 цифр в имени файла
        four_digit_pattern = re.compile(r'(\d{4})')

        for stl_path in tqdm(all_stl_files, desc="Сопоставление файлов"):
            match = four_digit_pattern.search(stl_path.stem)
            if not match:
                continue
            
            stl_number = match.group(1) # Извлекаем 4-значный номер
            
            # Ищем все изображения, начинающиеся с этого номера
            # (например, '0001_00.png', '0001_01.png', ...)
            image_paths = sorted(self.image_root.glob(f"{stl_number}_*.png"))
            
            for image_path in image_paths:
                self.paired_files.append((stl_path, image_path))
        
        print(f"\nНайдено {len(all_stl_files)} STL файлов.")
        print(f"Создано {len(self.paired_files)} пар (STL, Изображение) для обучения.")
        
        self.image_transform = T.Compose([
            T.Resize((image_size, image_size)), T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self): return len(self.paired_files)

    def __getitem__(self, idx):
        stl_path, image_path = self.paired_files[idx]
        points = load_stl_xyz_only(stl_path, self.num_points)
        if points is None: return None
        try:
            image = Image.open(image_path).convert("RGB")
            image_tensor = self.image_transform(image)
        except Exception: return None
        return torch.from_numpy(points), image_tensor

def paired_collate_fn(batch):
    """(Без изменений) Collate функция для нового датасета."""
    batch = list(filter(lambda x: x is not None, batch))
    if not batch: return None, None
    points, images = zip(*batch)
    return torch.stack(points), torch.stack(images)

# --- 2. АРХИТЕКТУРА МОДЕЛЕЙ ---

# 2.1 STL ЭНКОДЕР (PointNet++ из вашего кода)

# (ИСПРАВЛЕНО) Вспомогательные функции для PointNet++ в читаемом и рабочем виде
def farthest_point_sample(xyz, npoint):
    device = xyz.device
    B, N, C = xyz.shape
    centroids = torch.zeros(B, npoint, dtype=torch.long).to(device)
    distance = torch.ones(B, N).to(device) * 1e10
    farthest = torch.randint(0, N, (B,), dtype=torch.long).to(device)
    batch_indices = torch.arange(B, dtype=torch.long).to(device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch_indices, farthest, :].view(B, 1, 3)
        dist = torch.sum((xyz - centroid) ** 2, -1)
        mask = dist < distance
        distance[mask] = dist[mask]
        farthest = torch.max(distance, -1)[1]
    return centroids

def query_ball_point(radius, nsample, xyz, new_xyz):
    device = xyz.device
    B, N, C = xyz.shape
    _, S, _ = new_xyz.shape
    group_idx = torch.arange(N, dtype=torch.long, device=device).view(1, 1, N).repeat(B, S, 1)
    sqrdists = torch.sum((xyz.unsqueeze(1) - new_xyz.unsqueeze(2)) ** 2, -1)
    group_idx[sqrdists > radius ** 2] = N
    group_idx = group_idx.sort(dim=-1)[0][:, :, :nsample]
    group_first = group_idx[:, :, 0].view(B, S, 1).repeat(1, 1, nsample)
    mask = group_idx == N
    group_idx[mask] = group_first[mask]
    return group_idx

def index_points(points, idx):
    device = points.device
    B = points.shape[0]
    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1
    batch_indices = torch.arange(B, dtype=torch.long).to(device).view(view_shape).repeat(repeat_shape)
    new_points = points[batch_indices, idx, :]
    return new_points

class PointNetSetAbstraction(nn.Module):
    def __init__(self, npoint, radius, nsample, in_channel, mlp, group_all):
        super(PointNetSetAbstraction, self).__init__()
        self.npoint, self.radius, self.nsample, self.group_all = npoint, radius, nsample, group_all
        self.mlp_convs, self.mlp_bns = nn.ModuleList(), nn.ModuleList()
        last_channel = in_channel + 3
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            last_channel = out_channel

    def forward(self, xyz, points):
        if not self.group_all:
            new_xyz_idx = farthest_point_sample(xyz, self.npoint)
            new_xyz = index_points(xyz, new_xyz_idx)
            group_idx = query_ball_point(self.radius, self.nsample, xyz, new_xyz)
            grouped_xyz = index_points(xyz, group_idx)
            grouped_xyz -= new_xyz.unsqueeze(2)
            if points is not None:
                grouped_points = index_points(points, group_idx)
                features = torch.cat([grouped_xyz, grouped_points], dim=-1)
            else:
                features = grouped_xyz
        else:
            new_xyz = torch.zeros(xyz.shape[0], 1, 3, device=xyz.device)
            grouped_xyz = xyz.view(xyz.shape[0], 1, -1, 3)
            if points is not None:
                features = torch.cat([grouped_xyz, points.view(points.shape[0], 1, -1, points.shape[2])], dim=-1)
            else:
                features = grouped_xyz
        
        features = features.permute(0, 3, 2, 1)
        for conv, bn in zip(self.mlp_convs, self.mlp_bns):
            features = F.relu(bn(conv(features)))
        
        new_points = torch.max(features, 2)[0].permute(0, 2, 1)
        return new_xyz, new_points

class StlEncoder(nn.Module):
    """(ИСПРАВЛЕНО) Ваш класс Encoder, переименован для ясности."""
    def __init__(self, in_features=3, embedding_dim=256):
        super().__init__()
        # in_channel теперь правильно 0, так как у нас нет доп. фичей кроме xyz
        self.sa1 = PointNetSetAbstraction(npoint=512, radius=0.2, nsample=32, in_channel=in_features-3, mlp=[64, 64, 128], group_all=False)
        self.sa2 = PointNetSetAbstraction(npoint=128, radius=0.4, nsample=64, in_channel=128, mlp=[128, 128, 256], group_all=False)
        self.sa3 = PointNetSetAbstraction(npoint=None, radius=None, nsample=None, in_channel=256, mlp=[256, 512, 1024], group_all=True)
        self.fc1 = nn.Linear(1024, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.drop1 = nn.Dropout(0.4)
        self.fc_embedding = nn.Linear(512, embedding_dim)
    
    def forward(self, xyz):
        l1_xyz, l1_points = self.sa1(xyz, points=None)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        _, l3_points = self.sa3(l2_xyz, l2_points)
        x = l3_points.view(xyz.shape[0], -1)
        x = self.drop1(F.relu(self.bn1(self.fc1(x))))
        embedding = self.fc_embedding(x)
        return F.normalize(embedding, dim=1)

In [28]:
# ==============================================================================
# --- 2. КОНФИГУРАЦИЯ (ИЗМЕНЕННАЯ ВЕРСИЯ) ---
# ==============================================================================
# --- ПУТИ К ДАННЫМ ---
STL_DIR = Path("train/train_data/models")
IMAGE_DIR = Path("train/train_data/images")

# --- ПУТИ К ВХОДНЫМ И ВЫХОДНЫМ ФАЙЛАМ ---
# !!! ВАЖНО: Укажите путь к файлу с эмбеддингами, полученными после обучения с TripletLoss
PRECOMPUTED_STL_EMBEDDINGS_PATH = Path("precomputed_stl_embeddings.pt")

# Выходной файл с разметкой (остается без изменений)
METADATA_SAVE_PATH = Path("metadata_supcon.csv")

# --- ПАРАМЕТРЫ КЛАСТЕРИЗАЦИИ ПО ПОРОГУ ---
# Порог косинусного расстояния для объединения объектов в один кластер.
# Расстояние < порога -> один объект.
MERGE_THRESHOLD = 0.08

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Будут использоваться эмбеддинги из: {PRECOMPUTED_STL_EMBEDDINGS_PATH}")
print(f"Порог для слияния: {MERGE_THRESHOLD}")
print(f"Выходной файл с разметкой: {METADATA_SAVE_PATH}")

Будут использоваться эмбеддинги из: precomputed_stl_embeddings.pt
Порог для слияния: 0.08
Выходной файл с разметкой: metadata_supcon.csv


In [29]:
# ==============================================================================
# --- 3. ЗАГРУЗКА ПРЕД-ВЫЧИСЛЕННЫХ ЭМБЕДДИНГОВ (ИЗМЕНЕННАЯ ВЕРСИЯ) ---
# ==============================================================================
print("\n--- Этап 1: Загрузка пред-вычисленных STL эмбеддингов ---")

try:
    # Загружаем словарь {stem: tensor}
    stl_embeddings_dict_stems = torch.load(PRECOMPUTED_STL_EMBEDDINGS_PATH, map_location='cpu')
    print(f"Эмбеддинги успешно загружены из {PRECOMPUTED_STL_EMBEDDINGS_PATH}")
except FileNotFoundError:
    raise FileNotFoundError(f"Файл с эмбеддингами не найден: {PRECOMPUTED_STL_EMBEDDINGS_PATH}")

# --- Сопоставляем stem'ы с полными путями, так как они нужны далее ---
all_stl_files = sorted([f for f in STL_DIR.rglob('*.stl') if f.is_file()])
stem_to_path = {f.stem: str(f) for f in all_stl_files}

paths = []
embeddings_list = []
for stem, embedding in stl_embeddings_dict_stems.items():
    if stem in stem_to_path:
        paths.append(stem_to_path[stem])
        embeddings_list.append(embedding.numpy())
    else:
        print(f"Предупреждение: для stem='{stem}' не найден соответствующий STL файл в {STL_DIR}")

if not embeddings_list:
    raise ValueError("Не удалось найти эмбеддинги для файлов в указанной директории.")

# Преобразуем в матрицу для кластеризации
embedding_matrix = np.array(embeddings_list)
print(f"Найдено и сопоставлено {len(paths)} эмбеддингов.")

# L2 нормализация эмбеддингов перед кластеризацией
print(f"Нормализация {embedding_matrix.shape[0]} векторов...")
embedding_matrix = normalize(embedding_matrix, axis=1)

print("Эмбеддинги готовы для кластеризации.")


--- Этап 1: Загрузка пред-вычисленных STL эмбеддингов ---
Эмбеддинги успешно загружены из precomputed_stl_embeddings.pt
Найдено и сопоставлено 525 эмбеддингов.
Нормализация 525 векторов...
Эмбеддинги готовы для кластеризации.


In [30]:
# ==============================================================================
# --- 4. КЛАСТЕРИЗАЦИЯ ПО ПОРОГУ СХОДСТВА (ИЗМЕНЕННАЯ ВЕРСИЯ) ---
# ==============================================================================
from scipy.spatial.distance import cdist
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix

print(f"\n--- Этап 2: Поиск групп по порогу косинусного расстояния ({MERGE_THRESHOLD}) ---")

# 1. Вычисляем матрицу попарных косинусных расстояний
# Результат - матрица, где M[i, j] - расстояние между эмбеддингом i и j
print("Вычисление матрицы расстояний...")
distance_matrix = cdist(embedding_matrix, embedding_matrix, 'cosine')

# 2. Создаем матрицу смежности (Adjacency Matrix)
# True, если расстояние меньше порога (т.е. объекты похожи), иначе False
adjacency_matrix = distance_matrix < MERGE_THRESHOLD

# 3. Находим связанные компоненты в графе
# Это и есть наши кластеры. Каждый набор связанных вершин - один кластер.
print("Поиск связанных компонентов (кластеров)...")
graph = csr_matrix(adjacency_matrix)
n_clusters, cluster_labels = connected_components(
    csgraph=graph,
    directed=False, # Граф неориентированный
    return_labels=True # Нам нужны метки для каждого объекта
)

print(f"Кластеризация завершена.")
print(f"Найдено уникальных кластеров/концептов: {n_clusters}")

# 4. Создаем DataFrame с результатами (в том же формате, что и раньше)
df_clusters = pd.DataFrame({
    'stl_path': paths,
    'concept_id': cluster_labels # Метки от connected_components уже являются нашими concept_id
})

print("DataFrame с разметкой успешно создан.")


--- Этап 2: Поиск групп по порогу косинусного расстояния (0.08) ---
Вычисление матрицы расстояний...
Поиск связанных компонентов (кластеров)...
Кластеризация завершена.
Найдено уникальных кластеров/концептов: 392
DataFrame с разметкой успешно создан.


In [31]:
# ==============================================================================
# --- 5. СОЗДАНИЕ ЕДИНОГО METADATA ФАЙЛА ---
# ==============================================================================
print("\n--- Этап 3: Создание финального файла разметки ---")

all_files_data = []
four_digit_pattern = re.compile(r'(\d{4})')

# Добавляем информацию о STL файлах
for _, row in tqdm(df_clusters.iterrows(), total=len(df_clusters), desc="Обработка STL"):
    all_files_data.append({
        'filepath': row['stl_path'],
        'concept_id': row['concept_id'],
        'type': 'stl'
    })

# Находим соответствующие изображения и присваиваем им тот же concept_id
stl_path_to_concept_id = df_clusters.set_index('stl_path')['concept_id'].to_dict()

all_image_files = sorted([f for f in IMAGE_DIR.rglob('*.png') if f.is_file()])
all_stl_stems_map = {Path(p).stem: p for p in paths}


for image_path in tqdm(all_image_files, desc="Обработка изображений"):
    match = four_digit_pattern.search(image_path.stem)
    if not match: continue
    
    # Ищем STL файл, который породил это изображение, по номеру
    stl_number_str = match.group(1)
    
    # Находим полный stem STL файла (может иметь префиксы/суффиксы)
    # Это не самый надежный способ, но он основан на логике вашего датасета
    corresponding_stl_stem = next((s for s in all_stl_stems_map if stl_number_str in s), None)
    
    if corresponding_stl_stem:
        full_stl_path = all_stl_stems_map[corresponding_stl_stem]
        concept_id = stl_path_to_concept_id.get(full_stl_path)
        if concept_id is not None:
            all_files_data.append({
                'filepath': str(image_path),
                'concept_id': concept_id,
                'type': 'image'
            })

df_metadata = pd.DataFrame(all_files_data)

# Сохраняем результат
df_metadata.to_csv(METADATA_SAVE_PATH, index=False)

print(f"\nРазметка завершена! Файл сохранен в {METADATA_SAVE_PATH}")
print(f"Всего записей: {len(df_metadata)}")
print(f"Уникальных концептов: {df_metadata['concept_id'].nunique()}")
print("\nПример содержимого файла:")
print(df_metadata.head())


--- Этап 3: Создание финального файла разметки ---


Обработка STL:   0%|          | 0/525 [00:00<?, ?it/s]

Обработка изображений:   0%|          | 0/13650 [00:00<?, ?it/s]


Разметка завершена! Файл сохранен в metadata_supcon.csv
Всего записей: 14175
Уникальных концептов: 392

Пример содержимого файла:
                           filepath  concept_id type
0  train\train_data\models\0000.stl           0  stl
1  train\train_data\models\0001.stl           1  stl
2  train\train_data\models\0002.stl           2  stl
3  train\train_data\models\0003.stl           3  stl
4  train\train_data\models\0004.stl           4  stl


In [32]:
# ==============================================================================
# --- 6. ВИЗУАЛИЗАЦИЯ КЛАСТЕРОВ (ФИНАЛЬНАЯ ВЕРСИЯ С ВСТРОЕННЫМИ ИЗОБРАЖЕНИЯМИ) ---
# ==============================================================================
# Этот блок создаст ЕДИНЫЙ, АВТОНОМНЫЙ HTML-файл, в который будут встроены все
# рендеры. Это решает проблему с безопасностью браузеров при открытии локальных файлов.

# --- Установка и импорты ---
# Убедитесь, что установлены все библиотеки:
# !pip install matplotlib pillow

import trimesh
import cv2 as cv
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import io
import base64 # Библиотека для кодирования изображений

print("\n--- Этап 4: Создание автономного HTML-отчета для визуальной проверки ---")

# --- Конфигурация визуализации ---
OUTPUT_DIR = Path("cluster_visualization")
HTML_FILENAME = "cluster_report_standalone.html" # Новое имя, чтобы не перепутать
RENDER_SIZE = 200 # Размер превью-изображения

# Создаем папку для вывода, если ее нет
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# --- Функция рендеринга через Matplotlib (без изменений) ---
def render_stl_with_matplotlib(stl_path, size=200):
    try:
        mesh = trimesh.load(stl_path, force='mesh')
        if not hasattr(mesh, 'faces') or len(mesh.faces) == 0:
            raise ValueError("Сетка не содержит граней (faces) для отрисовки.")
        fig = plt.figure(figsize=(size/100, size/100), dpi=100)
        ax = fig.add_subplot(111, projection='3d')
        ax.axis('off'); ax.grid(False); ax.set_facecolor('white'); fig.patch.set_facecolor('white')
        ax.plot_trisurf(
            mesh.vertices[:, 0], mesh.vertices[:, 1], mesh.vertices[:, 2],
            triangles=mesh.faces, color='lightblue', edgecolor='k', linewidth=0.1
        )
        max_range = np.array([mesh.vertices[:,i].max()-mesh.vertices[:,i].min() for i in range(3)]).max() / 2.0
        mid = [ (mesh.vertices[:,i].max()+mesh.vertices[:,i].min()) * 0.5 for i in range(3)]
        ax.set_xlim(mid[0] - max_range, mid[0] + max_range)
        ax.set_ylim(mid[1] - max_range, mid[1] + max_range)
        ax.set_zlim(mid[2] - max_range, mid[2] + max_range)
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0, facecolor=fig.get_facecolor())
        plt.close(fig)
        buf.seek(0)
        pil_img = Image.open(buf)
        cv_img = cv.cvtColor(np.array(pil_img), cv.COLOR_RGBA2BGR)
        return cv_img
    except Exception:
        error_img = np.full((size, size, 3), 255, dtype=np.uint8)
        cv.putText(error_img, "Render", (size//2 - 40, size//2 - 10), cv.FONT_HERSHEY_SIMPLEX, 0.7, (150, 150, 150), 2)
        cv.putText(error_img, "Error", (size//2 - 30, size//2 + 20), cv.FONT_HERSHEY_SIMPLEX, 0.7, (150, 150, 150), 2)
        return error_img

# --- Находим кластеры с дубликатами ---
cluster_counts = df_clusters['concept_id'].value_counts()
duplicate_clusters_ids = cluster_counts[cluster_counts > 1].index.tolist()

if not duplicate_clusters_ids:
    print("Не найдено кластеров с количеством элементов > 1. Отчет не будет создан.")
else:
    print(f"Найдено {len(duplicate_clusters_ids)} кластеров с дубликатами. Начинаем генерацию HTML-отчета...")

    # --- HTML-шаблон (без изменений) ---
    html_content = """
    <!DOCTYPE html><html><head><title>Отчет по кластерам</title><style>
    body { font-family: sans-serif; margin: 20px; background-color: #f4f4f9; } h1 { text-align: center; color: #333; }
    .cluster { border: 1px solid #ccc; border-radius: 8px; margin-bottom: 20px; padding: 15px; background-color: #fff; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }
    .cluster h2 { margin-top: 0; color: #555; border-bottom: 2px solid #eee; padding-bottom: 10px; }
    .item-grid { display: flex; flex-wrap: wrap; gap: 15px; } .item { text-align: center; width: 200px; }
    .item img { border: 1px solid #ddd; border-radius: 4px; width: 100%; height: auto; background-color: #f0f0f0; }
    .item p { margin: 5px 0 0 0; font-size: 12px; word-wrap: break-word; color: #666; }
    </style></head><body><h1>Визуализация кластеров (потенциальных дубликатов)</h1>
    """

    for cid in tqdm(duplicate_clusters_ids, desc="Генерация отчета"):
        cluster_files = df_clusters[df_clusters['concept_id'] == cid]['stl_path'].tolist()
        html_content += f'<div class="cluster"><h2>Кластер ID: {cid} ({len(cluster_files)} элементов)</h2><div class="item-grid">'
        
        for file_path_str in cluster_files:
            file_path = Path(file_path_str)
            
            # 1. Генерируем изображение в памяти
            render_img = render_stl_with_matplotlib(file_path, size=RENDER_SIZE)
            
            # --- ИЗМЕНЕНИЕ: Кодируем изображение в Base64 для встраивания в HTML ---
            # Конвертируем массив OpenCV обратно в формат PNG (в памяти)
            _, buffer = cv.imencode('.png', render_img)
            # Кодируем байты в строку Base64
            b64_str = base64.b64encode(buffer).decode('utf-8')
            # Создаем Data URI
            img_src = f"data:image/png;base64,{b64_str}"
            
            # 2. Добавляем элемент в HTML, используя Data URI в качестве источника
            html_content += f"""
            <div class="item">
                <img src="{img_src}" alt="{file_path.name}">
                <p>{file_path.name}</p>
            </div>
            """
        
        html_content += '</div></div>\n'

    html_content += "</body></html>"

    # --- Сохраняем финальный HTML-файл ---
    report_path = OUTPUT_DIR / HTML_FILENAME
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
        
    print("\nОтчет успешно создан!")
    print(f"Откройте ЕДИНЫЙ файл в браузере: {report_path.resolve()}")


--- Этап 4: Создание автономного HTML-отчета для визуальной проверки ---
Найдено 68 кластеров с дубликатами. Начинаем генерацию HTML-отчета...


Генерация отчета:   0%|          | 0/68 [00:00<?, ?it/s]


Отчет успешно создан!
Откройте ЕДИНЫЙ файл в браузере: D:\AIIJC\cluster_visualization\cluster_report_standalone.html
